# Joint motion to foot motion

**Question:** How do reference frame and joint pose change the same velocity command's foot motion?

Two finite comparisons reuse [the one-link fixture](../jacobians_task_space.py) and [the two-joint fixture](../two_joint_planar_leg.py). They isolate local kinematics; they do not establish a gait or contact stability.

**Robot connection:** [foot_placement_diagnostics](../../spider/simulation.py) computes foot Jacobians and a joint-space update direction. [SupportAwareStanceController](../../spider/controllers.py) applies that direction to target angles only inside declared support. A velocity map, a task-error update, and a force-to-torque map use related Jacobians but have different units and purposes.

**Recorded learning:** Read the original [findings](../history/2026-08-09-jacobians-task-space/README.md) and [interaction record](../history/2026-08-09-jacobians-task-space/agent-log.md). No new interpretation or run result is claimed here.

In [ ]:
from pathlib import Path
import sys
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "spider" / "simulation.py").is_file())
sys.path.insert(0, str(ROOT))
import mujoco
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from lab import jacobians_task_space as frames, two_joint_planar_leg as leg
print("Setup only; comparisons have not run.")

## 1. Reference-frame comparison

**Edit here:** base angles and the joint pose. Keep gravity disabled to isolate geometry. Evaluate the analytic X Jacobian and a centered finite difference at each angle. This is an instantaneous finite comparison, not the historical moving-controller trace.

In [ ]:
base_angles = [0, 45, 90, 135, 180]
joint_angle = 0.0
rows = []
for degrees in base_angles:
    model = frames.make_model(degrees, gravity_z=0)
    data = mujoco.MjData(model)
    data.qpos[0] = joint_angle
    mujoco.mj_forward(model, data)
    site = model.site("foot").id
    rows.append((degrees, frames.x_jacobian(model, data, site),
                 frames.finite_difference_x_jacobian(model, data, site)))
comparison = pd.DataFrame(rows, columns=["base_degrees", "analytic_m_per_rad", "finite_difference_m_per_rad"])
display(comparison)
axis = comparison.plot(x="base_degrees", marker="o")
axis.set(xlabel="base pitch (degrees)", ylabel="foot X sensitivity (m/rad)")

## 2. Pose-dependent velocity map

**Edit here:** the two poses and the shared joint velocity. Evaluate each pose once. Compare `J(q) qdot` with MuJoCo's site velocity in the same world X-Z frame. The printed difference checks consistency; it does not prove the model describes a physical robot.

In [ ]:
poses = {name: q.copy() for name, q in leg.POSES.items()}
leg.QDOT = np.array([0.70, -0.35])
model = mujoco.MjModel.from_xml_string(leg.XML)
data = mujoco.MjData(model)
results = [leg.evaluate_pose(model, data, model.site("foot").id, name, q)
           for name, q in poses.items()]
fig, axis = plt.subplots()
for result in results:
    leg.print_result(result)
    x, z = result["x"]
    vx, vz = result["predicted_xdot"]
    axis.quiver(x, z, vx, vz, angles="xy", scale_units="xy", scale=1)
    axis.annotate(result["name"], (x, z))
axis.set(xlabel="world X (m)", ylabel="world Z (m)", title="Foot velocity arrows: 1 m/s drawn as 1 m")
axis.axis("equal")
axis.grid()

## Interpretation and next connection

Which change came from the reference frame, and which came from the pose? Explain why the same joint velocity can produce different foot velocities. Follow the measured residual into `foot_placement_diagnostics` before interpreting the robot's target correction.

For a force comparison, continue with [static support](static_support_boundary.ipynb). For reachability across a finite joint range, use [leg workspace](c1n_leg_workspace.ipynb). The original spatial viewers remain available through `python -m lab.jacobians_task_space --overlay` and `python -m lab.two_joint_planar_leg --viewer`. They are separate live views, not notebook replay.